# Evaluación ARC-Easy

Este notebook evalúa preguntas de ciencias con opciones múltiples. Selecciona la opción con mayor log-verosimilitud, y muestra precisión normal y normalizada por longitud. Requiere `torch`, `tiktoken` y `datasets`.

In [1]:
from pathlib import Path
import sys

from datasets import load_dataset
import tiktoken
import torch
import torch.nn.functional as F

ROOT = Path.cwd().resolve()
if not (ROOT / 'Foundation Model').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'Foundation Model'))
from Transformer_arquitectures import GPTModel

#model_name ='model.pth'
model_name = "gpt2-fineweb-124m-step-120000.pt"

CHECKPOINT = ROOT / 'Model Checkpoints' / model_name
assert CHECKPOINT.is_file(), f'No se encontró el checkpoint: {CHECKPOINT}'
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

default_config = {'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.0, 'qkv_bias': False}
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
config = {**default_config, **(checkpoint.get('config', {}) if isinstance(checkpoint, dict) else {})}
model = GPTModel(config).to(device).eval()
model.load_state_dict(state_dict)
tokenizer = tiktoken.get_encoding('gpt2')
print(f'Modelo cargado en {device}; contexto={config["context_length"]}')


/Users/oscarmanuelhernandezhernandez/Documents/ML_python/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo cargado en mps; contexto=512


In [2]:
def score_completion(prompt_ids, completion_ids):
    tokens = (prompt_ids + completion_ids)[-config['context_length']:]
    completion_start = max(1, len(tokens) - len(completion_ids))
    input_ids = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    with torch.inference_mode():
        logits = model(input_ids[:, :-1])
    log_probs = F.log_softmax(logits, dim=-1).gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)[0]
    completion_log_probs = log_probs[completion_start - 1:]
    return completion_log_probs.sum().item(), completion_log_probs.mean().item()

def evaluate(max_examples=100):
    dataset = load_dataset('allenai/ai2_arc', 'ARC-Easy', split='validation')
    if max_examples is not None:
        dataset = dataset.select(range(min(max_examples, len(dataset))))
    correct_raw = correct_norm = 0
    for number, example in enumerate(dataset, start=1):
        prompt = tokenizer.encode(f"Question: {example['question']}\nAnswer:")
        choices = example['choices']
        scores = [score_completion(prompt, tokenizer.encode(' ' + answer)) for answer in choices['text']]
        correct_index = choices['label'].index(example['answerKey'])
        correct_raw += max(range(len(scores)), key=lambda i: scores[i][0]) == correct_index
        correct_norm += max(range(len(scores)), key=lambda i: scores[i][1]) == correct_index
        if number % 100 == 0 or number == len(dataset):
            print(f'{number}/{len(dataset)}  acc={correct_raw / number:.4f}  acc_norm={correct_norm / number:.4f}')
    return {'model_name' : model_name, "Benchmark" : "Arc_Easy" , 'examples': len(dataset), 'acc': correct_raw / len(dataset), 'acc_norm': correct_norm / len(dataset)}

# Usa None para los 570 ejemplos de validación.
results = evaluate(max_examples=100)
results


100/100  acc=0.3400  acc_norm=0.2700


{'model_name': 'gpt2-fineweb-124m-step-120000.pt',
 'Benchmark': 'Arc_Easy',
 'examples': 100,
 'acc': 0.34,
 'acc_norm': 0.27}

In [ ]:
from pathlib import Path
import json

output_dir = ROOT / "benchmarks_results" / model_name 

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "benchmarks_result_Arc_easy.txt"
output_file.write_text(
    json.dumps(results, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

SyntaxError: invalid syntax (3471954014.py, line 4)